# Problema 1 – Clasificación: Detección de Fatiga Muscular en Ciclismo

**Workshop 2 – Machine Learning & Deep Learning Aplicado**  
**Universidad EAFIT – Introducción a la Inteligencia Artificial (2026-01)**

---

**Dataset:** Muscle Fatigue Cycling  
**Fuente:** HuggingFace – [YominE/Muscle_Fatigue_Cycling](https://huggingface.co/datasets/YominE/Muscle_Fatigue_Cycling)  
**Objetivo:** Clasificar el estado muscular del sujeto durante sprints en bicicleta: condición normal (0) vs. desgaste muscular (1).

## 0. Instalación de dependencias

In [ ]:
# Instalar librerías necesarias en Colab
!pip install datasets scikit-learn scipy seaborn matplotlib numpy pandas --quiet

## 1. Análisis Preliminar del Problema

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset

# Cargar dataset desde HuggingFace
print('Cargando dataset desde HuggingFace...')
hf_dataset = load_dataset('YominE/Muscle_Fatigue_Cycling')
df = hf_dataset['train'].to_pandas()

print(f'Shape del dataset: {df.shape}')
df.head()

In [ ]:
# Exploración inicial
print('=== Información general ===')
print(df.info())
print('\n=== Estadísticos descriptivos ===')
df.describe()

In [ ]:
# 1a. Preprocesamiento del target: etiqueta 2 → 1 (desgaste)
print('Distribución original del target:')
print(df['Target'].value_counts())

df['Target'] = df['Target'].replace(2, 1)

print('\nDistribución del target después del reemplazo (0=Normal, 1=Desgaste):')
print(df['Target'].value_counts())

In [ ]:
# 1b. Clasificación de tipos de variables
print('Tipos de variables en el dataset original:')
print('\n- Time: variable numérica continua (segundos). Representa el instante de muestreo.')
print('- Muscle_1 ... Muscle_8: variables numéricas continuas (mV). Señales EMG crudas de 8 canales.')
print('- Target: variable binaria categórica ordinal. 0 = condición normal, 1 = desgaste muscular.')
print()

# Frecuencia de muestreo (Hz)
if 'Time' in df.columns:
    dt = df['Time'].diff().median()
    fs = round(1.0 / dt)
    print(f'Frecuencia de muestreo detectada: {fs} Hz')
else:
    fs = 1000
    print(f'Frecuencia de muestreo asumida: {fs} Hz')

print('Columnas del dataset:', list(df.columns))
# Detectar canales: todas las columnas numéricas excepto Time y Target
exclude = {'time', 'target', 'label', 'class'}
channels = [c for c in df.columns if c.lower() not in exclude and df[c].dtype in ['float64', 'float32', 'int64', 'int32']]
# Quitar Target si quedó incluido por ser numérico
channels = [c for c in channels if 'target' not in c.lower() and 'label' not in c.lower()]
print(f'Canales EMG detectados: {channels}')

In [ ]:
# Visualización de una porción de las señales en el tiempo
n_plot = int(fs * 5)  # 5 segundos
fig, axes = plt.subplots(len(channels), 1, figsize=(14, 2.2 * len(channels)), sharex=True)

if len(channels) == 1:
    axes = [axes]

time_axis = df['Time'].values[:n_plot] if 'Time' in df.columns else np.arange(n_plot) / fs

for ax, ch in zip(axes, channels):
    ax.plot(time_axis, df[ch].values[:n_plot], linewidth=0.7, color='steelblue')
    ax.set_ylabel(ch, fontsize=9)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Tiempo (s)')
fig.suptitle('Señales EMG – primeros 5 segundos', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('senal_emg_5s.png', dpi=150, bbox_inches='tight')
plt.show()

print('''
Conclusiones sobre la inspección visual:
- Las señales EMG son oscilaciones de alta frecuencia centradas en cero,
  típicas de actividad eléctrica muscular.
- Se observa variabilidad en amplitud entre canales, lo que indica
  diferente nivel de activación en cada músculo.
- La señal es ruidosa por naturaleza; el procesamiento por ventanas
  es fundamental para extraer información estadística.
''')

## 2. Extracción de Características (Feature Engineering)

In [ ]:
from scipy.signal import welch

def extraer_caracteristicas(ventana: np.ndarray, fs: int = 1000) -> dict:
    """
    Extrae 7 características de tiempo y frecuencia para una ventana 1D.

    Características extraídas:
    - RMS  (Root Mean Square): energía promedio de la señal. Aumenta con mayor
      activación muscular.
    - VAR  (Varianza): dispersión de la señal. Relacionada con la irregularidad
      del disparo neuromotor bajo fatiga.
    - ZCR  (Zero Crossing Rate): número de cruces por cero. Disminuye con la
      fatiga al bajar la frecuencia de la señal.
    - MAV  (Mean Absolute Value): promedio del valor absoluto. Indicador de
      amplitud de la señal, similar al RMS pero más robusto al ruido.
    - POT  (Potencia espectral total): energía total en el dominio de la
      frecuencia. Integra la información de todas las frecuencias.
    - F_MEDIA (Frecuencia media): promedio ponderado de frecuencias. Desciende
      progresivamente con la fatiga muscular.
    - F_MEDIANA (Frecuencia mediana): frecuencia que divide la potencia
      espectral en dos mitades iguales. Biomarcador clásico de fatiga EMG.
    """
    # --- Dominio del tiempo ---
    rms      = np.sqrt(np.mean(ventana ** 2))
    var      = np.var(ventana)
    zcr      = int(np.sum(np.diff(np.sign(ventana)) != 0))
    mav      = np.mean(np.abs(ventana))

    # --- Dominio de la frecuencia ---
    freqs, psd = welch(ventana, fs=fs, nperseg=min(256, len(ventana)))
    pot_total  = float(np.sum(psd))
    denom      = np.sum(psd)
    f_media    = float(np.sum(freqs * psd) / denom) if denom > 0 else 0.0
    pot_acum   = np.cumsum(psd)
    idx_med    = int(np.searchsorted(pot_acum, pot_acum[-1] / 2))
    f_mediana  = float(freqs[min(idx_med, len(freqs) - 1)])

    return {
        'rms': rms, 'var': var, 'zcr': zcr, 'mav': mav,
        'pot': pot_total, 'f_media': f_media, 'f_mediana': f_mediana
    }

print('Función de extracción de características definida.')
print('Características por canal: rms, var, zcr, mav, pot, f_media, f_mediana (7 en total)')
print(f'Total de características: {len(channels)} canales × 7 = {len(channels) * 7} features')

In [ ]:
# Construcción del nuevo dataset por ventanas de 1 segundo
window_size = int(fs)  # 1000 muestras = 1 segundo
n_windows   = len(df) // window_size

print(f'Total de muestras: {len(df)}')
print(f'Tamaño de ventana: {window_size} muestras (1 segundo)')
print(f'Número de ventanas: {n_windows}')
print('Extrayendo características... (puede tardar unos minutos)')

filas = []
feat_names = ['rms', 'var', 'zcr', 'mav', 'pot', 'f_media', 'f_mediana']

for i in range(n_windows):
    inicio = i * window_size
    fin    = inicio + window_size
    ventana_df = df.iloc[inicio:fin]

    fila = {}
    for canal in channels:
        ventana = ventana_df[canal].values.astype(float)
        feats   = extraer_caracteristicas(ventana, fs=fs)
        for nombre, valor in feats.items():
            fila[f'{canal}_{nombre}'] = valor

    # Target de la ventana: valor más frecuente en esas 1000 muestras
    fila['target'] = int(ventana_df['Target'].mode()[0])
    filas.append(fila)

    if (i + 1) % 100 == 0 or i == n_windows - 1:
        print(f'  Ventana {i+1}/{n_windows} procesada...')

nuevo_df = pd.DataFrame(filas)
print(f'\nNuevo dataset creado: {nuevo_df.shape}')
nuevo_df.head()

In [ ]:
# Guardar el nuevo dataset (útil para no re-ejecutar la extracción)
nuevo_df.to_csv('features_emg.csv', index=False)
print('Dataset guardado como features_emg.csv')
print(f'Columnas: {list(nuevo_df.columns)}')

## 3. Análisis Exploratorio de Datos (EDA)

In [ ]:
# Estadísticos descriptivos del nuevo dataset
print('=== Estadísticos descriptivos del nuevo dataset ===')
nuevo_df.describe().T

In [ ]:
# Balance de clases
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

counts = nuevo_df['target'].value_counts().sort_index()
labels = ['Normal (0)', 'Desgaste (1)']

axes[0].bar(labels, counts.values, color=['steelblue', 'salmon'], edgecolor='black')
axes[0].set_title('Balance de clases (conteo)', fontweight='bold')
axes[0].set_ylabel('Número de ventanas')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=labels, autopct='%1.1f%%',
            colors=['steelblue', 'salmon'], startangle=90)
axes[1].set_title('Balance de clases (%)', fontweight='bold')

plt.tight_layout()
plt.savefig('balance_clases.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nClase 0 (Normal):   {counts.get(0, 0)} ventanas')
print(f'Clase 1 (Desgaste): {counts.get(1, 0)} ventanas')
ratio = counts.get(0,1) / counts.get(1,1)
print(f'Ratio 0/1: {ratio:.2f}')
print('\nInterpretación: Un ratio cercano a 1 indica clases bien balanceadas.')

In [ ]:
# Distribuciones de características representativas (RMS y frecuencia mediana)
rms_cols    = [c for c in nuevo_df.columns if c.endswith('_rms')]
fmed_cols   = [c for c in nuevo_df.columns if c.endswith('_f_mediana')]

fig, axes = plt.subplots(2, len(rms_cols), figsize=(3 * len(rms_cols), 8))

for i, (rms_c, fmed_c) in enumerate(zip(rms_cols, fmed_cols)):
    for clase, color in zip([0, 1], ['steelblue', 'salmon']):
        subset = nuevo_df[nuevo_df['target'] == clase]
        axes[0, i].hist(subset[rms_c], bins=25, alpha=0.6, color=color,
                        label=f'Clase {clase}', density=True)
        axes[1, i].hist(subset[fmed_c], bins=25, alpha=0.6, color=color,
                        label=f'Clase {clase}', density=True)
    axes[0, i].set_title(rms_c, fontsize=8)
    axes[1, i].set_title(fmed_c, fontsize=8)
    axes[0, i].legend(fontsize=7)
    axes[1, i].legend(fontsize=7)

axes[0, 0].set_ylabel('RMS – Densidad')
axes[1, 0].set_ylabel('F. Mediana – Densidad')
fig.suptitle('Distribuciones de RMS y Frecuencia Mediana por clase', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('distribuciones.png', dpi=150, bbox_inches='tight')
plt.show()

print('''
Interpretación:
- RMS más alto en clase 1 indicaría mayor activación muscular bajo fatiga.
- Frecuencia mediana más baja en clase 1 es el biomarcador clásico de fatiga EMG:
  bajo fatiga el reclutamiento de fibras rápidas decrece y la señal se desplaza
  hacia frecuencias bajas.
''')

In [ ]:
# Boxplots por clase para todas las características de un canal
canal_ref = channels[0]
feat_names_cols = [f'{canal_ref}_{f}' for f in ['rms', 'var', 'zcr', 'mav', 'pot', 'f_media', 'f_mediana']]

fig, axes = plt.subplots(1, len(feat_names_cols), figsize=(3.5 * len(feat_names_cols), 5))

for ax, col in zip(axes, feat_names_cols):
    nuevo_df.boxplot(column=col, by='target', ax=ax, notch=False,
                     patch_artist=True,
                     boxprops=dict(facecolor='steelblue', alpha=0.6))
    ax.set_title(col.split('_', 2)[-1], fontsize=9)
    ax.set_xlabel('Clase')

fig.suptitle(f'Boxplots por clase – {canal_ref}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

print('''
Interpretación:
- Una separación clara entre las cajas de clase 0 y clase 1 indica alta separabilidad
  y que esa característica será útil para el clasificador.
- Superposición de cajas sugiere baja separabilidad aislada (puede ser útil en combinación).
''')

In [ ]:
# Mapa de correlaciones
feature_cols = [c for c in nuevo_df.columns if c != 'target']

corr = nuevo_df[feature_cols].corr()

plt.figure(figsize=(max(12, len(feature_cols) // 2), max(10, len(feature_cols) // 2)))
sns.heatmap(corr, cmap='coolwarm', center=0, linewidths=0.3,
            xticklabels=True, yticklabels=True, annot=False)
plt.title('Mapa de correlación entre características', fontsize=13, fontweight='bold')
plt.xticks(fontsize=6, rotation=90)
plt.yticks(fontsize=6)
plt.tight_layout()
plt.savefig('correlacion.png', dpi=150, bbox_inches='tight')
plt.show()

print('''
Interpretación:
- Alta correlación entre características del mismo canal (ej. RMS y MAV) es esperada,
  ya que miden conceptos similares. Podrían eliminarse características redundantes.
- Baja correlación entre dominios de tiempo y frecuencia es deseable: aportan
  información complementaria al clasificador.
''')

## 4. Procesamiento de Datos

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Separar X e y
X = nuevo_df[feature_cols].values
y = nuevo_df['target'].values

# Verificar valores nulos
nulos = nuevo_df[feature_cols].isnull().sum().sum()
print(f'Valores nulos en X: {nulos}')
print(f'Valores infinitos en X: {np.isinf(X).sum()}')

# Reemplazar infinitos por NaN para que el imputer los maneje
X = np.where(np.isinf(X), np.nan, X)

# Split estratificado: 70% train / 15% val / 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'\nTrain: {X_train.shape[0]} muestras ({100*X_train.shape[0]/len(X):.1f}%)')
print(f'Val:   {X_val.shape[0]} muestras ({100*X_val.shape[0]/len(X):.1f}%)')
print(f'Test:  {X_test.shape[0]} muestras ({100*X_test.shape[0]/len(X):.1f}%)')
print('\nJustificación: Split 70/15/15 es estándar para datasets medianos.')
print('Stratify garantiza que la proporción de clases se mantenga en cada split.')

In [ ]:
# Pipeline de preprocesamiento (imputer + scaler)
preprocess_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

X_train_p = preprocess_pipeline.fit_transform(X_train)
X_val_p   = preprocess_pipeline.transform(X_val)
X_test_p  = preprocess_pipeline.transform(X_test)

print('Pipeline de preprocesamiento aplicado.')
print(f'Media post-escala (train): {X_train_p.mean():.6f} (≈ 0)')
print(f'Std  post-escala (train):  {X_train_p.std():.6f} (≈ 1)')

## 5. Entrenamiento y Comparación de Modelos

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import time

def evaluar_modelo(modelo, X_tr, y_tr, X_v, y_v, X_te, y_te):
    """Calcula métricas en train, val y test."""
    resultados = {}
    for nombre, X, y in [('Train', X_tr, y_tr), ('Val', X_v, y_v), ('Test', X_te, y_te)]:
        y_pred = modelo.predict(X)
        resultados[nombre] = {
            'Accuracy':  round(accuracy_score(y, y_pred), 4),
            'Precision': round(precision_score(y, y_pred, zero_division=0), 4),
            'Recall':    round(recall_score(y, y_pred, zero_division=0), 4),
            'F1':        round(f1_score(y, y_pred, zero_division=0), 4),
        }
    return resultados

resultados_globales = {}

print('Modelos y grillas de hiperparámetros definidos.')

In [ ]:
# ── kNN ──
print('Entrenando kNN...')
t0 = time.time()
param_knn = {'n_neighbors': [3, 5, 7, 11, 15], 'weights': ['uniform', 'distance'], 'p': [1, 2]}
knn_search = RandomizedSearchCV(KNeighborsClassifier(), param_knn, n_iter=10,
                                cv=3, scoring='f1', random_state=42, n_jobs=-1)
knn_search.fit(X_train_p, y_train)
knn_best = knn_search.best_estimator_
resultados_globales['kNN'] = evaluar_modelo(knn_best, X_train_p, y_train, X_val_p, y_val, X_test_p, y_test)
print(f'  Mejores params: {knn_search.best_params_}  ({time.time()-t0:.1f}s)')

In [ ]:
# ── Decision Tree ──
print('Entrenando Decision Tree...')
t0 = time.time()
param_dt = {'max_depth': [None, 5, 10, 20], 'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 5], 'criterion': ['gini', 'entropy']}
dt_search = RandomizedSearchCV(DecisionTreeClassifier(random_state=42), param_dt, n_iter=15,
                               cv=3, scoring='f1', random_state=42, n_jobs=-1)
dt_search.fit(X_train_p, y_train)
dt_best = dt_search.best_estimator_
resultados_globales['Decision Tree'] = evaluar_modelo(dt_best, X_train_p, y_train, X_val_p, y_val, X_test_p, y_test)
print(f'  Mejores params: {dt_search.best_params_}  ({time.time()-t0:.1f}s)')

In [ ]:
# ── Random Forest ──
print('Entrenando Random Forest...')
t0 = time.time()
param_rf = {'n_estimators': [100, 200, 300], 'max_depth': [None, 10, 20],
            'min_samples_split': [2, 5], 'max_features': ['sqrt', 'log2']}
rf_search = RandomizedSearchCV(RandomForestClassifier(random_state=42), param_rf, n_iter=12,
                               cv=3, scoring='f1', random_state=42, n_jobs=-1)
rf_search.fit(X_train_p, y_train)
rf_best = rf_search.best_estimator_
resultados_globales['Random Forest'] = evaluar_modelo(rf_best, X_train_p, y_train, X_val_p, y_val, X_test_p, y_test)
print(f'  Mejores params: {rf_search.best_params_}  ({time.time()-t0:.1f}s)')

In [ ]:
# ── Gradient Boosting ──
print('Entrenando Gradient Boosting...')
t0 = time.time()
param_gb = {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1, 0.2],
            'max_depth': [3, 5, 7], 'subsample': [0.7, 0.9, 1.0]}
gb_search = RandomizedSearchCV(GradientBoostingClassifier(random_state=42), param_gb, n_iter=12,
                               cv=3, scoring='f1', random_state=42, n_jobs=-1)
gb_search.fit(X_train_p, y_train)
gb_best = gb_search.best_estimator_
resultados_globales['Gradient Boosting'] = evaluar_modelo(gb_best, X_train_p, y_train, X_val_p, y_val, X_test_p, y_test)
print(f'  Mejores params: {gb_search.best_params_}  ({time.time()-t0:.1f}s)')

In [ ]:
# ── Deep Neural Network ──
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

print('Entrenando DNN...')
n_features = X_train_p.shape[1]

def build_dnn(units_1=128, units_2=64, units_3=32, dropout=0.3, lr=1e-3):
    model = keras.Sequential([
        layers.Input(shape=(n_features,)),
        layers.Dense(units_1, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Dropout(dropout),
        layers.Dense(units_2, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Dropout(dropout),
        layers.Dense(units_3, activation='relu'),
        layers.Dropout(dropout / 2),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

dnn_model = build_dnn()
dnn_model.summary()

early_stop = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)

history = dnn_model.fit(
    X_train_p, y_train,
    validation_data=(X_val_p, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)

# Predicciones DNN
def dnn_predict(X):
    return (dnn_model.predict(X, verbose=0).ravel() >= 0.5).astype(int)

class DNNWrapper:
    def predict(self, X): return dnn_predict(X)

resultados_globales['DNN'] = evaluar_modelo(DNNWrapper(), X_train_p, y_train, X_val_p, y_val, X_test_p, y_test)
print('DNN entrenado.')

In [ ]:
# Curvas de entrenamiento DNN
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'],     label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Pérdida (Loss)')
axes[0].set_xlabel('Época')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'],     label='Train Acc')
axes[1].plot(history.history['val_accuracy'], label='Val Acc')
axes[1].set_title('Exactitud (Accuracy)')
axes[1].set_xlabel('Época')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Curvas de entrenamiento – DNN', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('curvas_dnn.png', dpi=150, bbox_inches='tight')
plt.show()

print('''
Interpretación:
- Si val_loss diverge de train_loss → overfitting.
- Si ambas pérdidas se mantienen altas → underfitting.
- Early stopping garantiza que se use el mejor modelo de validación.
''')

In [ ]:
# Tabla comparativa de todos los modelos
filas_tabla = []
for modelo_nombre, splits in resultados_globales.items():
    for split_nombre, metricas in splits.items():
        fila = {'Modelo': modelo_nombre, 'Split': split_nombre}
        fila.update(metricas)
        filas_tabla.append(fila)

tabla = pd.DataFrame(filas_tabla)
tabla_pivot = tabla.pivot_table(index='Modelo', columns='Split',
                                values=['Accuracy', 'Precision', 'Recall', 'F1'])

print('=== Tabla comparativa de modelos ===')
display(tabla_pivot.round(4))

# Visualización: F1 en Test
f1_test = tabla[tabla['Split'] == 'Test'][['Modelo', 'F1']].sort_values('F1', ascending=False)

plt.figure(figsize=(9, 4))
bars = plt.barh(f1_test['Modelo'], f1_test['F1'], color='steelblue', edgecolor='black')
plt.xlabel('F1-Score (Test)')
plt.title('Comparación de modelos – F1-Score en Test', fontweight='bold')
plt.xlim(0, 1.05)
for bar, val in zip(bars, f1_test['F1']):
    plt.text(val + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center')
plt.tight_layout()
plt.savefig('comparacion_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Respuestas analíticas
mejor_modelo_nombre = f1_test.iloc[0]['Modelo']
print(f'=== Análisis de resultados ===')
print(f'\n1. Mejor modelo por F1-Score en Test: {mejor_modelo_nombre}')
print('''
2. Detección de overfitting/underfitting:
   - Overfitting: accuracy en Train >> accuracy en Val/Test.
     Decision Tree sin poda suele mostrar este comportamiento.
   - Underfitting: accuracy baja en todos los splits.
     kNN con k muy grande puede presentarlo.
   - Los modelos ensemble (RF, GB) tienden a generalizar mejor.

3. Modelo para producción:
   Se seleccionaría el modelo con mejor F1-Score en Test y menor
   brecha entre Train y Test (mejor generalización). Los modelos
   ensemble son preferibles por su robustez.
''')

## 6. Evaluación Final del Mejor Modelo

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# Selección del mejor modelo sklearn (excluimos DNN para re-entrenamiento simple)
modelos_sklearn = {
    'kNN': knn_best, 'Decision Tree': dt_best,
    'Random Forest': rf_best, 'Gradient Boosting': gb_best
}

f1_sklearn = {k: resultados_globales[k]['Test']['F1'] for k in modelos_sklearn}
mejor_nombre_sk = max(f1_sklearn, key=f1_sklearn.get)
mejor_modelo_sk = modelos_sklearn[mejor_nombre_sk]

print(f'Mejor modelo sklearn: {mejor_nombre_sk}')

# Re-entrenamiento con Train + Val
X_trainval = np.vstack([X_train_p, X_val_p])
y_trainval = np.concatenate([y_train, y_val])

mejor_modelo_sk.fit(X_trainval, y_trainval)
print('Re-entrenamiento con Train+Val completado.')

In [ ]:
# Predicción final en Test
y_pred_final = mejor_modelo_sk.predict(X_test_p)

print('=== Métricas finales en Test ===')
print(classification_report(y_test, y_pred_final, target_names=['Normal', 'Desgaste']))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred_final)
disp = ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Desgaste'])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Matriz de confusión – {mejor_nombre_sk}', fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Boxplots: características representativas diferenciando predicciones
df_test_results = pd.DataFrame(X_test_p, columns=feature_cols)
df_test_results['pred'] = y_pred_final
df_test_results['real'] = y_test

# Mostrar las 6 características con mayor importancia (si RF) o primeras 6
if hasattr(mejor_modelo_sk, 'feature_importances_'):
    importancias = pd.Series(mejor_modelo_sk.feature_importances_, index=feature_cols)
    top_feats = importancias.nlargest(6).index.tolist()
else:
    top_feats = feature_cols[:6]

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
axes = axes.ravel()

for ax, feat in zip(axes, top_feats):
    df_test_results.boxplot(column=feat, by='pred', ax=ax,
                            patch_artist=True,
                            boxprops=dict(facecolor='steelblue', alpha=0.5))
    ax.set_title(feat, fontsize=8)
    ax.set_xlabel('Predicción (0=Normal, 1=Desgaste)')

fig.suptitle('Boxplots de top características según predicción del modelo', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('boxplots_pred.png', dpi=150, bbox_inches='tight')
plt.show()

print('''
Respuesta: ¿Es un buen clasificador?
- Se considera bueno si F1 > 0.85 en Test.
- Posibles mejoras: aumentar ventanas con overlap, agregar características
  wavelet, usar más datos de sujetos distintos, o aplicar técnicas de
  selección de características para reducir redundancia.
''')

## 7. Prueba con Muestra Artificial

In [ ]:
# Generamos una muestra artificial con valores típicos de FATIGA (clase 1)
# Se eligen valores basados en los percentiles observados en el EDA para clase 1

np.random.seed(99)

# Obtener estadísticos de clase 1 para orientar la muestra
clase1_stats = nuevo_df[nuevo_df['target'] == 1][feature_cols].describe()

# Muestra artificial: media de clase 1 + pequeño ruido gaussiano
muestra_media = clase1_stats.loc['mean'].values
muestra_std   = clase1_stats.loc['std'].values * 0.1  # 10% de la desviación

muestra_artificial = muestra_media + np.random.randn(len(feature_cols)) * muestra_std
muestra_df = pd.DataFrame([muestra_artificial], columns=feature_cols)

# Preprocesar con el mismo pipeline
muestra_p = preprocess_pipeline.transform(muestra_df.values)

# Predicción
pred_muestra = mejor_modelo_sk.predict(muestra_p)[0]
pred_label   = 'DESGASTE MUSCULAR (1)' if pred_muestra == 1 else 'CONDICIÓN NORMAL (0)'

print('=== Prueba con muestra artificial ===')
print(f'Valores de la muestra (primeras 10 features):')
print(muestra_df.iloc[0, :10].round(5).to_string())
print(f'\n→ Predicción del modelo: {pred_label}')
print('''
Análisis:
La muestra fue construida con valores promedio de ventanas clasificadas como
desgaste muscular. Si el modelo predice correctamente "1", demuestra que ha
aprendido el patrón de fatiga: mayor RMS/MAV (mayor activación compensatoria)
y menor frecuencia mediana (desplazamiento espectral hacia bajas frecuencias).

El resultado tiene sentido fisiológicamente: bajo fatiga muscular, el sistema
nervioso recluta fibras adicionales (sube RMS) y la velocidad de conducción de
los potenciales disminuye (baja frecuencia mediana).
''')

In [ ]:
# Repetir con una muestra de condición NORMAL para contraste
clase0_stats = nuevo_df[nuevo_df['target'] == 0][feature_cols].describe()
muestra_normal = clase0_stats.loc['mean'].values + np.random.randn(len(feature_cols)) * (clase0_stats.loc['std'].values * 0.1)
muestra_normal_p = preprocess_pipeline.transform(pd.DataFrame([muestra_normal], columns=feature_cols).values)

pred_normal = mejor_modelo_sk.predict(muestra_normal_p)[0]
pred_label_normal = 'DESGASTE MUSCULAR (1)' if pred_normal == 1 else 'CONDICIÓN NORMAL (0)'

print(f'Predicción muestra artificial NORMAL: {pred_label_normal}')
print('\n=== Notebook completo. ===')